# Stage-1 Interpreter — slice-first de-risk run (caption → corpus → train → score)

Runs the CHEAP de-risk before committing the full caption budget: caption ~500 LUTs, build the
leakage-safe interpreter corpus, full-FT **Qwen2.5-0.5B-Instruct** (minutes on a T4 — does NOT need
the A100 the collapse-fix loop is using), and score on the `split_unit_id` holdout.

**GO/NO-GO:** the interpreter must beat a trivial always-`grade` baseline on `attribute_f1` +
route accuracy. If a 0.5B model on ~2.5k rows can't, stop before the full run (~2h + pennies spent,
not 15h + A100). CELL 1 prompts for `TFY_API_KEY` (teacher) + `HF_TOKEN` (read); the base URL is
pre-filled — press Enter to keep any value, or type a new one to fix a wrong entry.

In [ ]:
# CELL 1 - provision (clone, install, tokens, stage corpus for active_rows.jsonl)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    # [frontier] brings the openai SDK the teacher gateway needs; [sft] covers torch/transformers.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft,frontier]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# ALWAYS re-prompt (so a wrong/missing key is trivial to fix). Press Enter to KEEP the current value
# (from the environment or an uploaded .env); type a new value to overwrite it. TFY_BASE_URL is
# pre-filled with the known gateway.
import getpass
_DEFAULTS = {'TFY_BASE_URL': 'https://tfy.promptlens.trilogy.com/v1'}
for _n in ('HF_TOKEN', 'TFY_BASE_URL', 'TFY_API_KEY'):
    _cur = os.environ.get(_n) or _env_token(_n) or _DEFAULTS.get(_n, '')
    if not _cur:
        _hint = ' [required]'
    elif _n == 'TFY_BASE_URL':
        _hint = f' [Enter=keep {_cur}]'          # a URL is not secret; show it in full
    else:
        _hint = f' [Enter=keep ...{_cur[-4:]}]'  # token: show only the last 4 chars
    os.environ[_n] = getpass.getpass(f'{_n}{_hint}: ').strip() or _cur
# Honest readiness: BOTH TFY vars must be present (the old check only tested the base URL, which
# masked a missing TFY_API_KEY and let the caption cell fail with a hand-off).
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')),
      '| TFY ready:', bool(os.environ.get('TFY_BASE_URL')) and bool(os.environ.get('TFY_API_KEY')))

# The interpreter needs active_rows.jsonl (the out_of_scope refuse rows + split_unit_id source).
# We caption text-only (--no-image), so the full image/VQ corpus is not required; stage only if the
# jsonl is missing.
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('data/active_sft/active_rows.jsonl').is_file():
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
print('active_rows.jsonl:', pathlib.Path('data/active_sft/active_rows.jsonl').is_file())

In [ ]:
# CELL 2 - data: caption slice + clarify/out_of_gamut supplement + unified leakage-safe corpus
import subprocess, sys
LIMIT = 500          # slice size (LUTs). Full run later drops this.
# 2a. grade captions (text-only teacher; --workers for speed).
rc = subprocess.run([sys.executable, '-m', 'scripts.generate_captions',
                     '--limit', str(LIMIT), '--no-image', '--workers', '6']).returncode
assert rc == 0, 'caption gen failed / handed off (check TFY_BASE_URL + TFY_API_KEY)'
# 2b. clarify + out_of_gamut route supplement (full 3-way).
rc = subprocess.run([sys.executable, '-m', 'scripts.generate_route_supplement',
                     '--n-clarify', '150', '--n-gamut', '150']).returncode
assert rc == 0, 'route supplement failed / handed off'
# 2c. unify + stamp split_unit_id (the leakage fix).
rc = subprocess.run([sys.executable, '-m', 'scripts.build_interpreter_corpus']).returncode
assert rc == 0, 'corpus build failed'
import json
print(json.dumps(json.load(open('data/interpreter/interpreter_corpus_manifest.json')), indent=2))

In [ ]:
# CELL 3 - train the interpreter (full-FT Qwen2.5-0.5B-Instruct; minutes on a T4)
import subprocess, sys, glob
rc = subprocess.run([sys.executable, '-m', 'interpreter.train',
                     '--config', 'configs/candidate_interpreter.json',
                     '--run-id', 'interp_slice']).returncode
assert rc == 0, 'train failed (read the [interp] output above)'
ADAPTER = sorted(glob.glob('models/interpreter/interp_slice_*'))[-1]
print('trained:', ADAPTER)

In [ ]:
# CELL 4 - score on the untouched holdout; read METRIC + the GO/NO-GO components
import subprocess, sys, glob
ADAPTER = sorted(glob.glob('models/interpreter/interp_slice_*'))[-1]
subprocess.run([sys.executable, '-m', 'interpreter.score',
                '--config', 'configs/candidate_interpreter.json', '--adapter', ADAPTER])
print('\nGO/NO-GO: METRIC (unit-macro joint) and route_accuracy must beat a trivial always-grade\n'
      'baseline, and attribute_f1[real_lut] must be > 0. If not -> the approach is wrong; stop\n'
      'before the full ~2761-LUT caption run.')